# ERDES: inspect a small training-video sample

**Goal:** understand the actual video inputs before choosing baseline preprocessing.
This is a technical and visual inspection, not model training or a clinical diagnosis.
Displayed labels are the dataset's annotations.

Run this notebook independently, one cell at a time, on a **CPU** runtime. It selects
two training clips per subtype (eight total, about 4.3 MB) using a fixed selection rule.
The sample is deliberately balanced for inspection; it does not represent dataset prevalence
and is not a training subset or a performance estimate. No validation/test clips are inspected.

**Known limitation:** [paper v2, Data Record](https://arxiv.org/html/2508.04735v2#S3)
states that splitting was at clip level because patient identifiers were removed, so clips
from one patient may cross partitions. Actual patient-overlap counts are unavailable.
Filename prefixes are not documented patient IDs. Retain this limitation in future reports.

Source videos: [the authors' Hugging Face dataset](https://huggingface.co/datasets/pcvlab/erdes),
whose dataset card declares CC-BY-4.0. Cite Ozkut et al., *ERDES: A Benchmark Video Dataset
for Retinal Detachment and Macular Status Classification in Ocular Ultrasound*,
arXiv:2508.04735v2 (2026). Downloads below pin a revision and verify published file SHA256 values.


## 1. Check the runtime and choose storage

pandas handles tables, OpenCV (`cv2`) decodes video, and matplotlib displays frames.
The managed Colab runtime normally provides these libraries. If an import fails, stop and
follow [the environment guide](README.md) before adding packages. No GPU or PyTorch is needed.

Local inputs live under ignored `data/erdes/`; results go under ignored
`outputs/video_inspection/`. Colab uses `/content` when no project checkout is present.
Files on the remote runtime are separate from your local computer and are temporary.


In [1]:
import hashlib
import io
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import quote
from urllib.request import urlopen

import cv2
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Video, display

cwd = Path.cwd()
project_root = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
    None,
)
if project_root is None:
    if not Path("/content").is_dir():
        raise RuntimeError("Open this notebook from the project folder, or select a Colab kernel.")
    project_root = Path("/content")

data_dir = project_root / "data" / "erdes"
output_dir = project_root / "outputs" / "video_inspection"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
runtime = {"python": sys.version, "pandas": pd.__version__, "opencv": cv2.__version__, "matplotlib": matplotlib.__version__}
print(runtime)
print("Inputs:", data_dir)
print("Results:", output_dir)

def download_checked(url, path, expected_hash, algorithm="sha256", expected_bytes=None):
    if path.exists():
        content = path.read_bytes()
    else:
        with urlopen(url, timeout=90) as response:
            content = response.read()
    assert hashlib.new(algorithm, content).hexdigest() == expected_hash, f"Checksum mismatch: {path.name}"
    if expected_bytes is not None:
        assert len(content) == expected_bytes, f"Unexpected byte count: {path.name}"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    return content


{'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'pandas': '2.2.3', 'opencv': '5.0.0', 'matplotlib': '3.10.0'}
Inputs: /content/data/erdes
Results: /content/outputs/video_inspection


## 2. Select only training clips

Verify metadata and the official split archive from the same Zenodo release as the
previous notebooks. Match training paths to metadata and check **0 = non-RD, 1 = RD**.
We read only `train.csv` from the archive.

Within each subtype, sort clips by the SHA256 of `42:<clip_id>` and select the first two.
This fixed rule avoids selection based on visual appearance and gives the same sample
across runtimes without depending on pandas' random-sampling implementation.


In [2]:
release_url = "https://zenodo.org/records/18644370"
metadata_bytes = download_checked(
    f"{release_url}/files/erdes_metadata.csv?download=1", data_dir / "erdes_metadata.csv",
    "95828fb5bdffdbf8173b5bcfc99e8148", algorithm="md5",
)
archive_bytes = download_checked(
    f"{release_url}/files/splits.zip?download=1", data_dir / "splits.zip",
    "c5630e3f056ce358dcd384aaed946e64", algorithm="md5",
)
metadata = pd.read_csv(io.BytesIO(metadata_bytes))
assert len(metadata) == 5381 and metadata["file_path"].is_unique and metadata["clip_id"].is_unique
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    train_bytes = archive.read("splits/non_rd_vs_rd/train.csv")
train = pd.read_csv(io.BytesIO(train_bytes))
assert len(train) == 3873 and train["path"].is_unique
training = train.merge(metadata, left_on="path", right_on="file_path", how="left", validate="one_to_one")
assert training["clip_id"].notna().all(), "Training paths missing from metadata"
assert training["label"].map({0: "non_rd", 1: "rd"}).eq(training["diagnostic_class"]).all(), "Label mismatch"

inspection_seed = 42
subtype_order = ["normal", "pvd", "macula_intact", "macula_detached"]
chosen = []
for subtype in subtype_order:
    candidates = training.loc[training["subtype"].eq(subtype)].copy()
    candidates["selection_rank"] = candidates["clip_id"].map(
        lambda clip_id: hashlib.sha256(f"{inspection_seed}:{clip_id}".encode()).hexdigest()
    )
    assert len(candidates) >= 2
    chosen.append(candidates.sort_values("selection_rank").head(2))
sample = pd.concat(chosen, ignore_index=True).drop(columns="selection_rank")
assert len(sample) == 8 and sample["clip_id"].is_unique
assert set(sample["file_path"]).issubset(set(train["path"]))
display(sample[["clip_id", "diagnostic_class", "subtype", "frame_count", "fps", "width", "height", "duration_seconds"]])
print("PASS: eight training clips selected; no validation/test CSV was read")


,clip_id,diagnostic_class,subtype,frame_count,fps,width,height,duration_seconds
0,164267_03674,non_rd,normal,142,23.08,448,448,6.15
1,164267_02201,non_rd,normal,125,27.78,496,496,4.50
2,825315_00216,non_rd,pvd,90,30.00,432,432,3.00
3,825315_00468,non_rd,pvd,96,31.00,800,800,3.10
4,102769_00077,rd,macula_intact,174,25.00,448,448,6.96
5,258033_00019,rd,macula_intact,122,30.22,1040,1040,4.04
6,755384_00079,rd,macula_detached,52,28.57,496,496,1.82
7,405744_00139,rd,macula_detached,109,35.00,784,784,3.11


PASS: eight training clips selected; no validation/test CSV was read


## 3. Download the eight videos and verify their hashes

The immutable source revision and per-file hashes below were checked against the
[authors' file manifest](https://huggingface.co/api/datasets/pcvlab/erdes/revision/7867e2504ed3845a46ff7790a2f8c3ffcdc08947?blobs=true).
The expected hashes are Git LFS SHA256 values, not hashes inferred from our downloads.
Keep the selected clips under `data/erdes/clips/`; existing copies are verified and reused.
If a cached file differs, stop and investigate rather than overwriting it.


In [3]:
hf_revision = "7867e2504ed3845a46ff7790a2f8c3ffcdc08947"
hf_dataset = "https://huggingface.co/datasets/pcvlab/erdes"

clip_sources = {
    "164267_03674": {
        "sha256": "1986fa40aa2431598ffc858ef2442ae5fb16afa5a37dae047f71cc0d7687795c",
        "bytes": 264720
    },
    "164267_02201": {
        "sha256": "e6c627c62c701dd1b239e58cce71c6463ddc3fc7719f91487b870d5457ba495a",
        "bytes": 559116
    },
    "825315_00216": {
        "sha256": "0409107c7beed9035cf18a8ab81652c41fba462bacde6d7690b478d6f210712a",
        "bytes": 157320
    },
    "825315_00468": {
        "sha256": "29f290ad101cdb7e63388197c332e138bb300cb78049bad1b91bc877e8033737",
        "bytes": 641622
    },
    "102769_00077": {
        "sha256": "17c9e9676950096e55a318307be47d163d8c709d72f26fa8a2612129b8ce76d7",
        "bytes": 607206
    },
    "258033_00019": {
        "sha256": "c84306e773c40f3d6b1b5db8f64a03522c6fdfb9e28d125113dd6bccc3a631a6",
        "bytes": 941360
    },
    "755384_00079": {
        "sha256": "0b2ab744d20bef3b93a1fe99b1ae775fe3f19490d71a12904687ae9144007f7f",
        "bytes": 232445
    },
    "405744_00139": {
        "sha256": "ca3e8ae46e5bb51e0c698a3a93fcbd2617a6e31b4f5169731dad6aeea8088d6b",
        "bytes": 843198
    }
}

assert set(sample["clip_id"]) == set(clip_sources), "Selection differs from the recorded source manifest"
print(f"Selected download size: {sum(v['bytes'] for v in clip_sources.values()) / 1_000_000:.2f} MB")
provenance = []
for row in sample.itertuples(index=False):
    relative_path = PurePosixPath(row.file_path)
    assert not relative_path.is_absolute() and ".." not in relative_path.parts
    source = clip_sources[row.clip_id]
    url = f"{hf_dataset}/resolve/{hf_revision}/clips/{quote(row.file_path, safe='/')}"
    local_path = data_dir / "clips" / row.file_path
    download_checked(url, local_path, source["sha256"], expected_bytes=source["bytes"])
    provenance.append({
        "clip_id": row.clip_id, "file_path": row.file_path, "partition": "train",
        "dataset_label": int(row.label), "subtype": row.subtype, "url": url,
        "sha256": source["sha256"], "bytes": source["bytes"],
    })
    print(row.clip_id, "SHA256 PASS")


Selected download size: 4.25 MB
164267_03674 SHA256 PASS
164267_02201 SHA256 PASS
825315_00216 SHA256 PASS
825315_00468 SHA256 PASS
102769_00077 SHA256 PASS
258033_00019 SHA256 PASS
755384_00079 SHA256 PASS
405744_00139 SHA256 PASS


## 4. Decode each clip and compare it with metadata

Read every frame of these eight clips, count decoded frames, check dimensions and FPS,
and retain the first, middle, and last frames for display. The original videos are unchanged.
A failed check should be investigated before building a data loader.

This is a small sample check, not proof that all dataset videos are readable.
Decoder FPS may differ slightly due to rounding; we allow 1% or 0.1 FPS, whichever is larger.


In [4]:
decode_rows = []
preview_frames = {}
for row in sample.itertuples(index=False):
    capture = cv2.VideoCapture(str(data_dir / "clips" / row.file_path))
    frames = {}
    shapes = set()
    decoded_count = 0
    decoded_fps = float(capture.get(cv2.CAP_PROP_FPS))
    target_indices = {0, int(row.frame_count) // 2, int(row.frame_count) - 1}
    opened = capture.isOpened()
    try:
        while opened:
            ok, frame = capture.read()
            if not ok:
                break
            shapes.add(tuple(frame.shape[:2]))
            if decoded_count in target_indices:
                frames[decoded_count] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            decoded_count += 1
    finally:
        capture.release()
    count_matches = decoded_count == int(row.frame_count)
    dimensions_match = shapes == {(int(row.height), int(row.width))}
    fps_matches = abs(decoded_fps - float(row.fps)) <= max(0.1, 0.01 * float(row.fps))
    decode_rows.append({
        "clip_id": row.clip_id, "subtype": row.subtype, "opened": bool(opened),
        "expected_frames": int(row.frame_count), "decoded_frames": decoded_count,
        "expected_fps": float(row.fps), "decoded_fps": decoded_fps,
        "decoded_height_width": str(sorted(shapes)),
        "frame_count_matches": bool(count_matches), "dimensions_match": bool(dimensions_match),
        "fps_matches": bool(fps_matches),
        "technical_checks_pass": bool(opened and count_matches and dimensions_match and fps_matches),
    })
    preview_frames[row.clip_id] = frames
decode_report = pd.DataFrame(decode_rows)
decode_report.to_csv(output_dir / "decode_report.csv", index=False)
display(decode_report)
assert decode_report["technical_checks_pass"].all(), "Some video checks failed; inspect decode_report.csv before continuing"
print("PASS: all eight sampled videos decode and match metadata within the stated FPS tolerance")


,clip_id,subtype,opened,expected_frames,decoded_frames,expected_fps,decoded_fps,decoded_height_width,frame_count_matches,dimensions_match,fps_matches,technical_checks_pass
0,164267_03674,normal,True,142,142,23.08,23.08,"[(448, 448)]",True,True,True,True
1,164267_02201,normal,True,125,125,27.78,27.78,"[(496, 496)]",True,True,True,True
2,825315_00216,pvd,True,90,90,30.00,30.00,"[(432, 432)]",True,True,True,True
3,825315_00468,pvd,True,96,96,31.00,31.00,"[(800, 800)]",True,True,True,True
4,102769_00077,macula_intact,True,174,174,25.00,25.00,"[(448, 448)]",True,True,True,True
5,258033_00019,macula_intact,True,122,122,30.22,30.22,"[(1040, 1040)]",True,True,True,True
6,755384_00079,macula_detached,True,52,52,28.57,28.57,"[(496, 496)]",True,True,True,True
7,405744_00139,macula_detached,True,109,109,35.00,35.00,"[(784, 784)]",True,True,True,True


PASS: all eight sampled videos decode and match metadata within the stated FPS tolerance


## 5. Inspect frames and play the clips

Look for motion, crop position, orientation, brightness, borders, and remaining text or
device artifacts. These observations will guide preprocessing questions. Do not relabel
the videos or infer patient identity from these images.

The three snapshots provide a quick overview; use the playback controls to observe motion.
If an embedded player fails in your editor, open the downloaded MP4 in a compatible player;
browser playback and OpenCV decoding use different codec support.


In [ ]:
fig, axes = plt.subplots(len(sample), 3, figsize=(10, 3 * len(sample)))
for row_number, row in enumerate(sample.itertuples(index=False)):
    snapshots = sorted(preview_frames[row.clip_id].items())
    for column, ax in enumerate(axes[row_number]):
        ax.axis("off")
        if column < len(snapshots):
            frame_index, frame = snapshots[column]
            ax.imshow(frame)
            ax.set_title(f"Dataset label: {row.subtype} | frame {frame_index}", fontsize=9)
        if column == 0:
            ax.text(0, -0.05, row.clip_id, transform=ax.transAxes, fontsize=9)
fig.tight_layout()
fig.savefig(output_dir / "training_frame_examples.png", dpi=100)
plt.show()
plt.close(fig)

for row in sample.itertuples(index=False):
    print(row.clip_id, "| dataset subtype:", row.subtype)
    # Pass the path as data for IPython builds that inspect data before filename.
    display(Video(str(data_dir / "clips" / row.file_path), embed=True, width=320))


## 6. Save the inspection record and write your observations

Save source revision, hashes, selection rule, decoder results, and runtime versions.
The visual-review CSV starts blank and is preserved on reruns; fill it after watching clips.
Do not present automatic decoding checks as a completed human review.

Colab storage is temporary. Preserve the JSON report, decode CSV, frame overview, and your
completed notes outside the runtime. Clear embedded videos and frame images from the saved
notebook before committing it: those outputs contain video data and can be large.


In [8]:
notes_path = output_dir / "visual_review_notes.csv"
if not notes_path.exists():
    notes = sample[["clip_id", "subtype"]].copy()
    for column in ["motion_notes", "crop_orientation_notes", "brightness_border_artifact_notes", "questions"]:
        notes[column] = ""
    notes["review_status"] = "pending"
    notes.to_csv(notes_path, index=False)

report = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "metadata_release": release_url,
    "metadata_sha256": hashlib.sha256(metadata_bytes).hexdigest(),
    "split_archive_sha256": hashlib.sha256(archive_bytes).hexdigest(),
    "training_csv_sha256": hashlib.sha256(train_bytes).hexdigest(),
    "video_source": hf_dataset, "video_revision": hf_revision,
    "source_license_declaration": "cc-by-4.0 (author dataset card)",
    "paper": "https://arxiv.org/html/2508.04735v2",
    "selection_rule": "Within each subtype, rank by SHA256 of <seed>:<clip_id>; take first two",
    "inspection_seed": inspection_seed, "sample_count": len(sample), "partition": "train",
    "runtime": runtime, "clips": provenance,
    "technical_checks": decode_report.to_dict(orient="records"),
    "manual_review": "See visual_review_notes.csv; automatic checks do not complete human review",
    "limitations": [
        "Only eight training clips inspected; not a dataset-wide corruption or duplicate audit",
        "Paper documents clip-level splitting and possible same-patient overlap across partitions",
        "This is not model training, a performance estimate, or clinical validation",
    ],
}
report_path = output_dir / "inspection_summary.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("Saved:", report_path)
print("Saved:", output_dir / "decode_report.csv")
print("Saved:", output_dir / "training_frame_examples.png")
print("Fill in:", notes_path)


Saved: /content/outputs/video_inspection/inspection_summary.json
Saved: /content/outputs/video_inspection/decode_report.csv
Saved: /content/outputs/video_inspection/training_frame_examples.png
Fill in: /content/outputs/video_inspection/visual_review_notes.csv


## Saved inspection observations

All eight sample clips passed hash and decoding checks (910 frames total). Source sizes
in this sample range from 432×432 to 1040×1040. The sampled frames show changes in view
position/internal image features, brightness, and black margins. Fixed-size, temporally
ordered inputs are therefore a reasonable first engineering step. These observations do
not establish label correctness or an optimal preprocessing method.

Review method: agent inspection of sampled stills and adjacent-frame pixel-change metrics;
full video playback was not independently reviewed by the agent. Per-clip notes and measurements
are saved locally in ignored `outputs/video_inspection/visual_review_notes.csv` and
`observations_summary.json`. The executed notebook is archived there too.

**Next:** run [baseline_preparation.ipynb](baseline_preparation.ipynb) to verify the proposed
input transformation before the first GPU training pilot.


## Next: baseline preparation

The saved review summary above records the agent's observations and review method.
Per-clip notes, measurements, and the executed notebook are preserved under ignored
`outputs/video_inspection/`; these do not establish label correctness or patient independence.

Run [baseline_preparation.ipynb](baseline_preparation.ipynb) next. It checks uniform frame
sampling, grayscale, square padding, resizing, scaling, and model input shape. The proposed
first GPU pilot is described in [BASELINE_PLAN.md](../docs/BASELINE_PLAN.md). No model has
been trained; final test evaluation remains reserved.
